In [1]:
import numpy as np
import pandas as pd
import utils
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
import itertools
import os
from scipy.stats import qmc
from typing import Callable,Any
from scipy.spatial.transform import Rotation,Slerp
from elastica.modules import BaseSystemCollection,Constraints,Forcing,Damping,CallBacks
from elastica.rod.cosserat_rod import CosseratRod
from elastica.boundary_conditions import OneEndFixedBC,ConstraintBase
from elastica.external_forces import GravityForces
from elastica.dissipation import AnalyticalLinearDamper
from elastica.callback_functions import CallBackBaseClass,defaultdict
from elastica.timestepper.symplectic_steppers import PositionVerlet
from elastica.timestepper import integrate
import warnings
warnings.simplefilter('ignore')

In [2]:
def coner(d, angle_deg):
    v=d[2]
    v = v / np.linalg.norm(v)
    theta = np.deg2rad(angle_deg)
    if abs(v[0]) < 0.9:
        a = np.array([1,0,0])
    else:
        a = np.array([0,1,0])
    u = np.cross(v, a)
    u /= np.linalg.norm(u)
    w = np.cross(v, u)
    phi = np.random.uniform(0, 2 * np.pi)
    vec = (np.cos(theta)*v + np.sin(theta)*(np.cos(phi)*u + np.sin(phi)*w))
    return np.array([d[0],d[1],vec])

def doe_looper(tgt_dir,scenario,case=1,base_radius=2.43,density=1.23e-6,youngs_modulus=0.003605,twist=0,endvectr=10,length=163.32,intr='d',pfix='',csv=1,html=0):

    print('\n\nStarting for iter=',scenario)

    # Simulator
    class Simulator(
        BaseSystemCollection,Constraints,Forcing,Damping,CallBacks
    ):
        pass

    # Data input
    df=pd.read_csv('../data/2001/w'+str(case)+'.csv')
    pts=df.iloc[10:,2:].reset_index(drop=True)
    # length=utils.length(pts.values)
    simulator=Simulator()
    n_elements=100 
    base_length=length 
    base_radius=base_radius 
    start_pos=df.iloc[5,2:].values.astype(float)
    direction=np.array([0.0,1.0,0.0]) 
    normal=np.array([0.0,0.0,1.0])     
    density=density 
    youngs_modulus=youngs_modulus
    shear_modulus=youngs_modulus/(2.0*(1.0+0.49))

    # Rod def
    tube=CosseratRod.straight_rod(
        n_elements=n_elements,
        start=start_pos,
        direction=direction,
        normal=normal,
        base_length=base_length,
        base_radius=base_radius,
        density=density,
        youngs_modulus=youngs_modulus,
        shear_modulus=shear_modulus
    )

    # BCs
    # grav
    simulator.append(tube)
    simulator.add_forcing_to(tube).using(
        GravityForces,
        acc_gravity=np.array([0.0,0.0,-9.81e-3])
    )

    # damp
    dmp_const=0.3
    dt=1e-3
    simulator.dampen(tube).using(
        AnalyticalLinearDamper,
        damping_constant=dmp_const,
        time_step=dt
    )
    print('Added dampner to the simulation')

    # Fixed BC
    simulator.constrain(tube).using(
        OneEndFixedBC,
        constrained_position_idx=(0,),
        constrained_director_idx=(0,)
    )
    print('Added FixedBC')

    # movin end
    class EndpointKinematicConstraint(ConstraintBase):
        def __init__(self,
                    node_idx: int,
                    elem_idx: int,
                    target_position_function: Callable[[float],np.ndarray],
                    target_director_function: Callable[[float],np.ndarray],
                    target_velocity_function: Callable[[float],np.ndarray],
                    target_omega_function: Callable[[float],np.ndarray],
                    **kwargs):
            super().__init__(**kwargs)
            self.node_idx=node_idx
            self.elem_idx=elem_idx        
            self.pos_func=target_position_function
            self.dir_func=target_director_function
            self.vel_func=target_velocity_function
            self.omg_func=target_omega_function

        def constrain_values(self,system: Any,time: float) -> None:
            target_pos=self.pos_func(time)
            target_dir=self.dir_func(time)
            system.position_collection[...,self.node_idx]=target_pos
            system.director_collection[...,self.elem_idx]=target_dir
            
        def constrain_rates(self,system: Any,time: float) -> None:
            target_vel=self.vel_func(time)
            target_omg=self.omg_func(time)
            system.velocity_collection[...,self.node_idx]=target_vel
            system.omega_collection[...,self.elem_idx]=target_omg

    class TrajectoryRamp:
        def __init__(self,
                    start_pos: np.ndarray,
                    end_pos: np.ndarray,
                    start_dir: np.ndarray,
                    end_dir: np.ndarray,
                    ramp_time: float):
            
            self.start_pos=start_pos
            self.end_pos=end_pos
            self.pos_diff=end_pos - start_pos        
            self.ramp_time=ramp_time
            self.const_vel=self.pos_diff / ramp_time
            key_rots=Rotation.from_matrix([start_dir,end_dir])
            key_times=[0,ramp_time]
            self.slerp=Slerp(key_times,key_rots)
            self.const_omg=np.zeros((3,))
            
        def _get_ramp_factor(self,time: float) -> float:
            if time < 0:
                return 0.0
            elif time > self.ramp_time:
                return 1.0
            else:
                return time / self.ramp_time

        def get_position(self,time: float) -> np.ndarray:
            factor=self._get_ramp_factor(time)
            return self.start_pos + self.pos_diff * factor

        def get_director(self,time: float) -> np.ndarray:
            clamped_time=max(0,min(time,self.ramp_time))
            return self.slerp(clamped_time).as_matrix()

        def get_velocity(self,time: float) -> np.ndarray:
            if 0 < time <= self.ramp_time:
                return self.const_vel
            else:
                return np.zeros((3,))

        def get_omega(self,time: float) -> np.ndarray:
            if 0 < time <= self.ramp_time:
                return self.const_omg
            else:
                return np.zeros((3,))
            
    class TrajectoryCircle:
        def __init__(self,center: np.ndarray,radius: float,freq: float):
            self.center=center
            self.radius=radius
            self.omega_val=2.0 * np.pi * freq

        def get_position(self,time: float) -> np.ndarray:
            x=self.center + self.radius * np.cos(self.omega_val * time)
            y=self.center + self.radius * np.sin(self.omega_val * time)
            return np.array([x,y,self.center])
        
        def get_velocity(self,time: float) -> np.ndarray:
            vx=-self.radius * self.omega_val * np.sin(self.omega_val * time)
            vy=self.radius * self.omega_val * np.cos(self.omega_val * time)
            return np.array([vx,vy,0.0])

        def get_director(self,time: float) -> np.ndarray:
            return np.identity(3)

        def get_omega(self,time: float) -> np.ndarray:
            return np.zeros((3,))
        
    # Define targets
    init_pos=tube.position_collection[...,-1].copy()
    init_dir=tube.director_collection[...,-1].copy()
    tgt_pos=df.iloc[6,2:].values
    theta=np.deg2rad(twist)
    tgt_dir[0]=(tgt_dir[0]*np.cos(theta)+np.cross(tgt_dir[2],tgt_dir[0])*np.sin(theta)+tgt_dir[2]*np.dot(tgt_dir[2],tgt_dir[0])*(1-np.cos(theta)))
    tgt_dir[1]=(tgt_dir[1]*np.cos(theta)+np.cross(tgt_dir[2],tgt_dir[1])*np.sin(theta)+tgt_dir[2]*np.dot(tgt_dir[2],tgt_dir[1])*(1-np.cos(theta)))
    ramp_time=350

    trajectory=TrajectoryRamp(
        init_pos,tgt_pos,init_dir,tgt_dir,ramp_time
    )

    simulator.constrain(tube).using(
        EndpointKinematicConstraint,
        node_idx=-1,
        elem_idx=-1,
        target_position_function=trajectory.get_position,
        target_director_function=trajectory.get_director,
        target_velocity_function=trajectory.get_velocity,
        target_omega_function=trajectory.get_omega
    )

    # cb
    class TubeCallback(CallBackBaseClass):
        def __init__(self,step_skip: int,callback_params: dict):
            CallBackBaseClass.__init__(self)
            self.every=step_skip
            self.callback_params=callback_params

        def make_callback(self,system: Any,time: int,current_step: int):
            if current_step % self.every == 0:
                self.callback_params["time"].append(time)
                self.callback_params["step"].append(current_step)
                self.callback_params["position"].append(system.position_collection.copy())
                self.callback_params["directors"].append(system.director_collection.copy())
                self.callback_params["length"].append(system.rest_lengths.copy())
                self.callback_params["radius"].append(system.radius.copy())
                self.callback_params["velocity"].append(system.velocity_collection.copy())
                self.callback_params["avg_velocity"].append(system.compute_velocity_center_of_mass())
                self.callback_params["com"].append(system.compute_position_center_of_mass())
                self.callback_params["curvature"].append(system.kappa.copy())
                return
    cb_data=defaultdict(list)
    dt_intrvl=100
    simulator.collect_diagnostics(tube).using(
        TubeCallback,
        step_skip=dt_intrvl,
        callback_params=cb_data
    )

    # finalize
    simulator.finalize()
    print('Simulator finalized')

    # solver
    timestepper=PositionVerlet()
    final_time=750 
    total_steps=int(final_time/dt)

    print("Running simulation...")
    integrate(timestepper,simulator,final_time,total_steps)
    print("Simulation finished.")

    # plotters
    position_history=np.array(cb_data["position"])
    final_rod_position=position_history[-1]

    df=pd.read_csv('../data/2001/w'+str(case)+'.csv')
    prt=df.iloc[10:,2:].reset_index(drop=True)
    p0=df.iloc[5,2:].values.astype(float)   
    p1=df.iloc[6,2:].values.astype(float)   
    u0=df.iloc[8,2:].values.astype(float)*df.iloc[2,1]   
    u1=df.iloc[9,2:].values.astype(float)*df.iloc[3,1]   
    siml=pd.DataFrame(final_rod_position.T,columns=['X','Y','Z'])

    if intr=='d':
        fig=utils.plotter(prt.values,p0,p1,u0,u1,x_label='Prototype',spl1=siml.values,spl1_label='Simulated',cne=6,intr='d')
        a=df.iloc[59,2:]
        b=siml.iloc[50]
        fig.add_trace(go.Scatter3d(x=[a[0],b[0]],y=[a[1],b[1]],z=[a[2],b[2]],mode='lines',name='Error',line=dict(color='red',width=6,dash='longdash')))
        # fig.show()
        if html==1:
            if not os.path.exists(pfix+'html/'):
                os.makedirs(pfix+'html/')
            # pio.write_html(fig,pfix+'html/p'+str(psit)+'-case'+str(case)+'.html')

    if intr=='s':
        fig,ax1,ax2,ax3,ax4=utils.plotter(prt.values,p0,p1,u0,u1,x_label='Prototype',spl1=siml.values,spl1_label='Simulated',cne=6,intr='s')
        ax4.plot([a[0],b[0]],[a[1],b[1]],[a[2],b[2]],label='Error',linestyle='--')
        ax4.legend()
        fig.show()

    if csv==1:
        if not os.path.exists(pfix+'data/'):
            os.makedirs(pfix+'data/')
        siml.to_csv(pfix+'data/w'+str(case)+'_'+scenario+'.csv',index=False)
    
    print('Task completed')

In [3]:
params = ['twist','youngs_modulus','base_radius','density','coner_angle','length']
levels = [-0.10, -0.05, 0.0, 0.05, 0.10]
baseline = {'twist':0,'youngs_modulus':0.003605,'base_radius':2.43,'density':1.23e-6,'coner_angle':0,'length':162.32}
rows = []
rows.append({
    **baseline,
    "scenario": "baseline"
})
for p in params:
    for l in levels:
        if l == 0.0:
            continue 
        row = baseline.copy()
        if p in ['twist','coner_angle']:
            if abs(l)==0.05:
                row[p]=5*np.sign(l)
            elif abs(l)==0.10:
                row[p]=10*np.sign(l)
        else:
            row[p]=baseline[p]*(1+l)
        row["scenario"] = f"{p}_{int(l*100)}"
        rows.append(row)
doe_df = pd.DataFrame(rows)
doe_df

,twist,youngs_modulus,base_radius,density,coner_angle,length,scenario
0,0.0,0.003605,2.4300,0.000001,0.0,162.320,baseline
1,-10.0,0.003605,2.4300,0.000001,0.0,162.320,twist_-10
2,-5.0,0.003605,2.4300,0.000001,0.0,162.320,twist_-5
3,5.0,0.003605,2.4300,0.000001,0.0,162.320,twist_5
4,10.0,0.003605,2.4300,0.000001,0.0,162.320,twist_10
5,0.0,0.003244,2.4300,0.000001,0.0,162.320,youngs_modulus_-10
6,0.0,0.003425,2.4300,0.000001,0.0,162.320,youngs_modulus_-5
7,0.0,0.003785,2.4300,0.000001,0.0,162.320,youngs_modulus_5
8,0.0,0.003966,2.4300,0.000001,0.0,162.320,youngs_modulus_10
9,0.0,0.003605,2.1870,0.000001,0.0,162.320,base_radius_-10


In [4]:
base_doe = doe_df.copy()
cases = [1, 2, 3, 4, 5]
all_dfs = []
for case in cases:
    temp = base_doe.copy()
    temp.insert(0, 'case', case)
    df = pd.read_csv(f'../data/2001/w{case}.csv')
    tgt_dir = utils.dir_unitvectr(df.iloc[9, 2:].values).T
    temp['tgt_dir'] = [coner(tgt_dir, angle) for angle in temp['coner_angle']]
    all_dfs.append(temp)
final_df = pd.concat(all_dfs, ignore_index=True)
final_df

,case,twist,youngs_modulus,base_radius,density,coner_angle,length,scenario,tgt_dir
0,1,0.0,0.003605,2.43,0.000001,0.0,162.320,baseline,"[[-0.6760472895516252, 0.0, -0.736858237580269..."
1,1,-10.0,0.003605,2.43,0.000001,0.0,162.320,twist_-10,"[[-0.6760472895516252, 0.0, -0.736858237580269..."
2,1,-5.0,0.003605,2.43,0.000001,0.0,162.320,twist_-5,"[[-0.6760472895516252, 0.0, -0.736858237580269..."
3,1,5.0,0.003605,2.43,0.000001,0.0,162.320,twist_5,"[[-0.6760472895516252, 0.0, -0.736858237580269..."
4,1,10.0,0.003605,2.43,0.000001,0.0,162.320,twist_10,"[[-0.6760472895516252, 0.0, -0.736858237580269..."
...,...,...,...,...,...,...,...,...,...
120,5,0.0,0.003605,2.43,0.000001,10.0,162.320,coner_angle_10,"[[0.42560769620071365, 0.0, -0.904907779243123..."
121,5,0.0,0.003605,2.43,0.000001,0.0,146.088,length_-10,"[[0.42560769620071365, 0.0, -0.904907779243123..."
122,5,0.0,0.003605,2.43,0.000001,0.0,154.204,length_-5,"[[0.42560769620071365, 0.0, -0.904907779243123..."
123,5,0.0,0.003605,2.43,0.000001,0.0,170.436,length_5,"[[0.42560769620071365, 0.0, -0.904907779243123..."


In [5]:
final_df.to_csv('wiper_ofat10p/data/doe_df.csv',index=False)

In [6]:
def run_row(i):
    return doe_looper(
        scenario=final_df.scenario.iloc[i],
        case=final_df.case.iloc[i],
        twist=final_df.twist.iloc[i],
        youngs_modulus=final_df.youngs_modulus.iloc[i],
        base_radius=final_df.base_radius.iloc[i],
        density=final_df.density.iloc[i],
        tgt_dir=final_df.tgt_dir.iloc[i],
        length=final_df.length.iloc[i],
        intr='n',
        pfix='wiper_ofat10p\\'
    )

In [7]:
from joblib import Parallel, delayed

Parallel(n_jobs=-1, backend="loky", verbose=10)(
    delayed(run_row)(i) for i in range(len(final_df))
)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 20 concurrent workers.
[Parallel(n_jobs=-1)]: Done   1 tasks      | elapsed:  2.9min
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:  3.1min
[Parallel(n_jobs=-1)]: Done  21 tasks      | elapsed:  5.8min
[Parallel(n_jobs=-1)]: Done  32 tasks      | elapsed:  6.2min
[Parallel(n_jobs=-1)]: Done  45 tasks      | elapsed:  9.1min
[Parallel(n_jobs=-1)]: Done  58 tasks      | elapsed:  9.5min
[Parallel(n_jobs=-1)]: Done  73 tasks      | elapsed: 12.5min
[Parallel(n_jobs=-1)]: Done  99 out of 125 | elapsed: 16.0min remaining:  4.2min
[Parallel(n_jobs=-1)]: Done 112 out of 125 | elapsed: 18.6min remaining:  2.2min
[Parallel(n_jobs=-1)]: Done 125 out of 125 | elapsed: 20.3min finished


[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]